# GameTheory 16j — Screening et loyer d'information

**Navigation** : [<< 16e-Players-LLM](GameTheory-16e-LLM-Players-Othman-Sandholm-Python.ipynb) | [Index GameTheory](README.md) | [Capstone 21 -- Choix social >>](GameTheory-21-Capstone-Choix-Social-Python.ipynb)

**Le coût mesuré de l'asymétrie d'information** dans le problème classique de *screening*
à sélection adverse (Maskin–Riley, Myerson).

À la famille **16 (Mécanismes)**, à côté de 16i (Appariage stable) : un principal
(employeur) offre un contrat à un agent (salarié) dont il **ne connaît pas le type**.
On mesure trois choses :

1. le **loyer d'information** que le principal doit verser au type haut pour que les deux
   types s'autosélectionnent honnêtement ;
2. la **distorsion** du contrat du type haut (on le fait travailler *moins* que l'optimum) ;
3. le **coût total de l'asymétrie** = surplus avec information complète − surplus avec
   screening, et sa **décomposition** en loyer + perte d'efficacité.

Toutes les valeurs sont **calculées** dans le carnet (numpy, CPU seul), pas énoncées.


## 1. Le modèle

Un salarié est de l'un de deux **types** (productivités) :

- type **bas** $\theta_L = 1$, type **haut** $\theta_H = 2$ ;
- probabilité $p$ que le salarié soit du type haut.

Le principal propose un contrat $(a, t)$ : un niveau d'effort $a$ et une rémunération $t$.

| | Formule | Rôle |
|---|---|---|
| Production | $V(\theta, a) = \theta \, a$ | ce que l'effort $a$ d'un type $\theta$ rapporte au principal |
| Coût d'effort | $C(\theta, a) = \dfrac{a^2}{2\theta}$ | ce que l'effort $a$ coûte au type $\theta$ |

Le type haut trouve l'effort **moins coûteux** ($C$ décroît en $\theta$) : c'est la condition
de **croisement simple** (single-crossing) qui rend le screening possible. L'agent a une
utilité $U(\theta) = t - C(\theta, a)$ ; le principal un surplus $S(\theta) = V(\theta, a) - t$.

> Le type est **non observable** par le principal : c'est l'asymétrie d'information.


In [1]:
import numpy as np

# --- Types et probabilité ---
theta_L = 1.0
theta_H = 2.0
p_H     = 0.5     # probabilité que le salarié soit du type haut

# --- Production et coût d'effort ---
def V(th, a):  return th * a
def C(th, a):  return a**2 / (2.0 * th)

print(f"theta_L = {theta_L}, theta_H = {theta_H}, p_H = {p_H}")
print(f"coût d'effort du type haut  C(2, a=2) = {C(theta_H, 2.0)}")
print(f"coût d'effort du type bas   C(1, a=2) = {C(theta_L, 2.0)}")
print("(le type haut trouve l'effort moins cher : croisement simple)")

theta_L = 1.0, theta_H = 2.0, p_H = 0.5
coût d'effort du type haut  C(2, a=2) = 1.0
coût d'effort du type bas   C(1, a=2) = 2.0
(le type haut trouve l'effort moins cher : croisement simple)


### Lecture

À effort égal ($a=2$), le type haut paie un coût $1$ là où le type bas en paie $2$.
C'est cette **asymétrie de coût** que le mécanisme va exploiter : en proposant le bon
contrat, on fait en sorte que chaque type *choisisse de lui-même* le sien.


## 2. Premier optimum — le principal connaît le type (benchmark)

Sans asymétrie, le principal fixe pour **chaque** type l'effort qui maximise son surplus
$S(\theta) = \theta a - a^2/(2\theta)$, et paie juste le coût (l'agent ne garde **aucun
loyer** : l'agent de participation est liant, $U=0$).

Le maximum en $a$ donne $a^*(\theta) = \theta^2$ (dérivée nulle : $\theta - a/\theta = 0$).


In [2]:
# --- Premier optimum (information complète) ---
a_L_1F = theta_L**2
a_H_1F = theta_H**2
t_L_1F = C(theta_L, a_L_1F)          # loyer = coût  ->  loyer d'information = 0
t_H_1F = C(theta_H, a_H_1F)
S_L_1F = V(theta_L, a_L_1F) - t_L_1F
S_H_1F = V(theta_H, a_H_1F) - t_H_1F

print("PREMIER OPTIMUM (le principal voit le type) :")
print(f"  type bas : a* = {a_L_1F:.2f},  t = {t_L_1F:.3f},  surplus = {S_L_1F:.3f}")
print(f"  type haut: a* = {a_H_1F:.2f},  t = {t_H_1F:.3f},  surplus = {S_H_1F:.3f}")
print(f"  (aucun loyer d'information : l'agent est payé à juste son coût)")

PREMIER OPTIMUM (le principal voit le type) :
  type bas : a* = 1.00,  t = 0.500,  surplus = 0.500
  type haut: a* = 4.00,  t = 4.000,  surplus = 4.000
  (aucun loyer d'information : l'agent est payé à juste son coût)


### Lecture

Avec information complète, le type haut travaille à $a^* = 4$ et le principal en retire
un surplus de **4.00**. C'est le **plafond** : tout ce qui manque plus bas est dû à
l'asymétrie. On garde ce chiffre en tête.


## 3. Second optimum — le principal ne voit pas le type (screening)

Le principal propose un **menu** de deux contrats. Les deux contraintes d'un menu
séparateur optimal :

- **Participation** du type bas liant : $U_L = 0$ (on ne paie pas de loyer au type bas) ;
- **Auto-sélection** : le type bas ne doit pas préférer le contrat du type haut.

Le résultat classique (croisement simple) : **le type bas reçoit l'effort du premier
optimum** (non déformé), et **le type haut voit son effort abaissé**. Son contrat porte
alors un **loyer d'information** — le supplément qu'il doit recevoir pour préférer son
propre contrat à celui du type bas.

En exprimant tout en fonction de $a_H$ via ces deux contraintes, le problème du principal
se réduit à :
$$\max_{a_H}\; \theta_H a_H - \frac{a_H^2}{2\theta_L}
\quad\Rightarrow\quad a_H^{2B} = \theta_H\,\theta_L .$$


In [3]:
# --- Second optimum (screening) ---
a_L_2B = theta_L**2                       # type bas : NON déformé (= 1er optimum)
a_H_2B = theta_H * theta_L                # type haut : DÉFORMÉ (abaissé)

# loyer d'information du type haut
U_H = a_H_2B**2 / 2.0 * (1.0/theta_L - 1.0/theta_H)

t_L_2B = C(theta_L, a_L_2B)               # U_L = 0
t_H_2B = U_H + C(theta_H, a_H_2B)         # t_H = loyer + coût
S_L_2B = V(theta_L, a_L_2B) - t_L_2B
S_H_2B = V(theta_H, a_H_2B) - t_H_2B

print("SECOND OPTIMUM (screening, le type est caché) :")
print(f"  type bas :  a = {a_L_2B:.2f}  (non déformé),  surplus = {S_L_2B:.3f}")
print(f"  type haut : a = {a_H_2B:.2f}  (abaissé vs {a_H_1F:.2f}), surplus = {S_H_2B:.3f}")
print(f"  LOYER D'INFORMATION du type haut  U_H = {U_H:.3f}")
print(f"  rémunération type haut t_H = {t_H_2B:.3f}  (= loyer {U_H:.3f} + coût {C(theta_H, a_H_2B):.3f})")

SECOND OPTIMUM (screening, le type est caché) :
  type bas :  a = 1.00  (non déformé),  surplus = 0.500
  type haut : a = 2.00  (abaissé vs 4.00), surplus = 2.000
  LOYER D'INFORMATION du type haut  U_H = 1.000
  rémunération type haut t_H = 2.000  (= loyer 1.000 + coût 1.000)


### Lecture

Le type haut ne travaille plus à $4$ mais à $a_H = 2$ : c'est la **distorsion** du
contrat du type haut. En échange, le principal doit lui verser un **loyer
d'information** $U_H = 1.00$ — le prix à payer pour qu'il ne se fasse pas passer pour
le type bas. Son surplus tombe de $4.00$ à $2.00$.


## 4. Le coût de l'asymétrie, et sa décomposition

Le **coût de l'asymétrie d'information** est l'écart entre le surplus du premier optimum
(§2) et celui du second (§3), pour le type haut :
$$\text{coût} = S_H^{1F} - S_H^{2B} .$$

On le **décompose** en deux parts :

- le **loyer d'information** $U_H$ payé au type haut ;
- la **perte d'efficacité** due à la distorsion de son effort
  ($a_H$ abaissé de $4$ à $2$) :
  $S_H^{1F} - \big(V(\theta_H, a_H^{2B}) - C(\theta_H, a_H^{2B})\big)$.


In [4]:
# --- Coût de l'asymétrie + décomposition (type haut) ---
cost_asym_H  = S_H_1F - S_H_2B
rent_H       = U_H
distortion_H = S_H_1F - (V(theta_H, a_H_2B) - C(theta_H, a_H_2B))

print("COÛT DE L'ASYMÉTRIE (par salarié de type haut) :")
print(f"  coût total            = {cost_asym_H:.3f}")
print(f"    dont loyer d'info   = {rent_H:.3f}   ({100*rent_H/cost_asym_H:.0f} %)")
print(f"    dont distorsion     = {distortion_H:.3f}   ({100*distortion_H/cost_asym_H:.0f} %)")
assert abs(rent_H + distortion_H - cost_asym_H) < 1e-9, "décomposition incohérente"

# coût attendu (moyenne sur les deux types)
cost_expected = p_H * cost_asym_H + (1 - p_H) * 0.0
print(f"  coût attendu (p_H = {p_H}) = {cost_expected:.3f}")

COÛT DE L'ASYMÉTRIE (par salarié de type haut) :
  coût total            = 2.000
    dont loyer d'info   = 1.000   (50 %)
    dont distorsion     = 1.000   (50 %)
  coût attendu (p_H = 0.5) = 1.000


### Lecture — le résultat central

Avec $\theta_L=1, \theta_H=2$, le coût de l'asymétrie se **partage à parts égales** :
**1.00 de loyer d'information** + **1.00 de perte par distorsion** = **2.00** de coût.
L'asymétrie d'information a un prix double : on paie l'agent pour son information, **et**
on sous-utilise sa productivité. Le type bas, lui, n'est pas affecté (son contrat reste
l'optimum, sans loyer) : tout le coût repose sur le type haut.


## 5. Sensibilité — le coût croît avec l'écart de type

Que se passe-t-il quand les deux types sont **plus différents** ? On fixe
$\theta_L = 1$ et on fait varier $\theta_H$, en recalculant le coût de l'asymétrie
du type haut pour chaque écart.


In [5]:
# --- Sensibilité : coût d'asymétrie en fonction de l'écart de type ---
print(f"{'theta_H':>8} | {'a_H 1F':>8} | {'a_H 2B':>8} | {'loyer':>8} | {'distorsion':>10} | {'coût total':>10}")
print("-" * 62)
for thH in [1.25, 1.5, 2.0, 2.5, 3.0]:
    aH1 = thH**2
    aH2 = thH * theta_L
    rent = aH2**2/2.0 * (1/theta_L - 1/thH)
    S1F  = V(thH, aH1) - C(thH, aH1)
    S2B  = V(thH, aH2) - (rent + C(thH, aH2))
    dist = S1F - (V(thH, aH2) - C(thH, aH2))
    print(f"{thH:>8.2f} | {aH1:>8.2f} | {aH2:>8.2f} | {rent:>8.3f} | {dist:>10.3f} | {S1F-S2B:>10.3f}")

 theta_H |   a_H 1F |   a_H 2B |    loyer | distorsion | coût total
--------------------------------------------------------------
    1.25 |     1.56 |     1.25 |    0.156 |      0.039 |      0.195
    1.50 |     2.25 |     1.50 |    0.375 |      0.188 |      0.562
    2.00 |     4.00 |     2.00 |    1.000 |      1.000 |      2.000
    2.50 |     6.25 |     2.50 |    1.875 |      2.812 |      4.688
    3.00 |     9.00 |     3.00 |    3.000 |      6.000 |      9.000


### Lecture

Le coût de l'asymétrie **croît de façon convexes** avec l'écart de type : $0.20$ pour
$\theta_H=1.25$, $2.00$ pour $\theta_H=2$, $9.00$ pour $\theta_H=3$. Plus les types
sont différents, plus le loyer d'information à verser au type haut est gros — c'est le
résultat qu'on attendait, **mesuré** plutôt que supposé.


## 6. Exercices

### Exercice 1 — Trois types
Étendez le calcul à trois types $\theta \in \{1, 1.5, 2\}$ (probabilités
$1/3$ chacun). Calculez le **loyer d'information** du type du milieu.

*Indice : dans un menu à $n$ types, seul le type le plus bas a un loyer nul ; chaque type
supérieur cumule les écarts de coût avec le type juste en dessous.*


In [6]:
# TODO étudiant — loyer d'information du type du milieu (3 types)
theta_mid = 1.5
# calculez a_H_2B du type milieu puis son loyer U_mid
loyer_type_milieu = None  # TODO étudiant
print("Exercice à compléter : loyer du type milieu =", loyer_type_milieu)

Exercice à compléter : loyer du type milieu = None


### Exercice 2 — Vérifier l'auto-sélection
Vérifiez numériquement que, au second optimum, le **type bas** ne préfère pas le contrat
du type haut (c'est la contrainte d'auto-sélection qui fixe le loyer).

*Indice : comparez l'utilité du type bas s'il prenait le contrat haut, $t_H -
C(\theta_L, a_H)$, à son utilité sur son propre contrat ($0$).*


In [7]:
# TODO étudiant — le type bas préfère-t-il le contrat haut ?
utilite_bas_si_contrat_haut = None  # TODO étudiant : t_H_2B - C(theta_L, a_H_2B)
auto_selection_respectee = None     # TODO étudiant : True si <= 0
print("Exercice à compléter : utilité du bas sur contrat haut =",
      utilite_bas_si_contrat_haut, "-> auto-sélection respectée ?", auto_selection_respectee)

Exercice à compléter : utilité du bas sur contrat haut = None -> auto-sélection respectée ? None


### Exercice 3 — Le point 50/50
Pour $\theta_L = 1$, trouvez la valeur de $\theta_H$ pour laquelle le **loyer
d'information** et la **perte par distorsion** sont **égaux** (le partage 50/50 vu au
§4).

*Indice : c'est le $\theta_H$ où `rent_H == distortion_H`. (Ici $\theta_H = 2$ —
vérifiez-le, ne l'affirmez pas.)*


In [8]:
# TODO étudiant — theta_H où loyer == distorsion
theta_5050 = None  # TODO étudiant
print("Exercice à compléter : theta_H du partage 50/50 =", theta_5050)

Exercice à compléter : theta_H du partage 50/50 = None


## 7. Ce que ce carnet établit

- Le **screening** n'est pas sans coût : il fait payer à l'agent informé un **loyer
  d'information** **et** sous-utilise sa productivité (distorsion).
- Ce coût se **décompose** de façon exacte en loyer + distorsion — ici à parts égales
  ($1.00 + 1.00 = 2.00$), ce qui n'est **pas** un hasard général mais une conséquence des
  paramètres choisis.
- Le coût **croît convexe** avec l'écart de type : l'asymétrie d'information coûte d'autant
  plus cher que les types sont différents.

C'est le fondement des mécanismes de §16 (conception de mécanismes) : tout contrat qu'on
propose à un agent informé doit payer ce loyer, et le concevoir, c'est minimiser ce coût.
